# Script to map external files to SalishSeaCast grid and create the respective netcdf file

## Importing

In [2]:
import xarray as xr
import numpy as np
import os
from tqdm import tqdm
from salishsea_tools import grid_tools as gt
import math
 

## Loading datasets

In [3]:
# Daily dimensions loading
ds0 = xr.open_dataset('/data/ibougoudis/MOAD/files/inputs/jan_apr.nc')
mask = ds0['Diatom'][0].to_numpy()

# Obtaining lat and lon
mesh = xr.open_dataset('/home/sallen/MEOPAR/grid/mesh_mask202108.nc')
lat = mesh.nav_lat
lon = mesh.nav_lon

lat = lat.where(mask>0)
lon = lon.where(mask>0)

lat = np.broadcast_to(lat,(len(ds0.time_counter),len(ds0.y),len(ds0.x)))
lon = np.broadcast_to(lon,(len(ds0.time_counter),len(ds0.y),len(ds0.x)))

dayofyear = np.tile(np.arange(0,len(ds0.time_counter)//len(np.unique(ds0.time_counter.dt.year))), len(np.unique(ds0.time_counter.dt.year)))
dayofyear = np.reshape(np.repeat(dayofyear, len(ds0.x)*len(ds0.y)), (len (ds0.time_counter), len(ds0.y), len(ds0.x)))
dayofyear = np.where(~np.isnan(lat), dayofyear, np.nan)


## Calculations

In [4]:
def calculation(k):

    for j in tqdm(folders): 

        # Test for the proper date matching
        test = xr.open_dataset(path+j)
        if test.time_counter.dt.date[12] != ds0.time_counter[k].dt.date:
            print (ds0.time_counter[k].dt.date.values)

        inter, grid = gt.build_matrix(weight, (path+j))

        for i in range(0,24):

            u_hourly[i] =  gt.use_matrix(path+j, inter, grid, 'u_wind', i)
            v_hourly[i] =  gt.use_matrix(path+j, inter, grid, 'v_wind', i)
    
    

        wind_speed[k] = np.sqrt(u_hourly.mean('hour')**2 + v_hourly.mean('hour')**2).where(mask>0)
        wind_direction_1[k] = (np.arctan2(v_hourly,u_hourly) * (180/np.pi) + 180).mean('hour').where(mask>0)
        wind_direction_2[k] = ((270 - np.degrees(np.arctan2(v_hourly,u_hourly))) % 360).mean('hour').where(mask>0)
        wind_velocity[k] = (u_hourly*np.sin(math.radians(29)) + v_hourly*np.cos(math.radians(29))).mean('hour').where(mask>0)

        
        k = k + 1
    
    return k


## File creation

In [5]:
def file_creation(variable, name):

    temp = variable.to_dataset(name=name)
    temp.to_netcdf(path='/data/ibougoudis/MOAD/files/jan_apr_wind.nc', mode='a', encoding={name:{"zlib": True, "complevel": 9}})
    temp.close()
    

## Creation of datasets

In [6]:
# Coordinates
coords_hourly = dict(hour=np.arange(0,24), y=ds0.y, x=ds0.x)
coords = dict(time_counter= ds0.time_counter, y=ds0.y, x=ds0.x)

# k is the general counter, j the daily one, i the hourly one
k = 0

# Hourly arrays
u_hourly = xr.DataArray(coords=coords_hourly, dims = ['hour', 'y', 'x'])
v_hourly = xr.DataArray(coords=coords_hourly, dims = ['hour', 'y', 'x'])

# Daily arrays
wind_speed = xr.DataArray(coords=coords, dims = ['time_counter', 'y', 'x'])
wind_direction_1 = xr.DataArray(coords=coords, dims = ['time_counter', 'y', 'x'])
wind_direction_2 = xr.DataArray(coords=coords, dims = ['time_counter', 'y', 'x'])
wind_velocity = xr.DataArray(coords=coords, dims = ['time_counter', 'y', 'x'])


In [18]:
path = ('/results/forcing/atmospheric/continental2.5/nemo_forcing/')
folders = [x for x in os.listdir(path) if ((x[12:14]=='01' or (x[12:14]=='02' and x[15:17] < '29') or x[12:14]=='03' or x[12:14] == '04') and (x[7:11] < '2026'))]
folders.sort()
folders = folders[2:] # Cut the first two days in the folder
weight = ('/home/sallen/MEOPAR/grid/weights-continental2.5-hrdps_202108_23feb23onward.nc')
k = calculation(k)

100%|██████████| 307/307 [12:23<00:00,  2.42s/it]


In [ ]:
len(folders)

307

## Data collection

In [ ]:
# Four loops, one for each case

# First loop (15 Feb 2007 - 30 Apr 2011, different mask)
path = ('/results/forcing/atmospheric/GEM2.5/gemlam/')
folders = [x for x in os.listdir(path) if (x[13:15]=='01' or (x[13:15]=='02' and x[16:18] < '29') or x[13:15]=='03' or x[13:15] == '04') and (x[8:12] < '2013')]
folders.sort()
folders = [i for i in folders if i[18]=='.'] # Removing four double files
folders.insert(0,folders[0]) # Only for January
folders.insert(0,folders[0]) # Only for January
weight = ('/home/sallen/MEOPAR/grid/weights-gem2.5-gemlam_201702_pre22sep11.nc')
k = calculation(k)

# Second loop (15 Feb 2012 - 30 Apr 2014, different mask)
path = ('/results/forcing/atmospheric/GEM2.5/gemlam/')
folders = [x for x in os.listdir(path) if (x[13:15]=='01' or (x[13:15]=='02' and x[16:18] < '29') or x[13:15]=='03' or x[13:15] == '04') and (x[8:12] > '2012')]
folders.sort()
weight = ('/home/sallen/MEOPAR/grid/weights-gem2.5-gemlam_201702_22sep11onward.nc')
k = calculation(k)

# Third loop (15 Feb 2015 - 22 Feb 2023)
path = ('/results/forcing/atmospheric/GEM2.5/operational/')
folders = [x for x in os.listdir(path) if ((x[10:12]=='01' or (x[10:12]=='02' and x[13:15] <'29') or x[10:12]=='03' or x[10:12] == '04'))]
folders.sort()
folders = [i for i in folders if i[15]=='.'] # Removing one double file
weight = ('/home/sallen/MEOPAR/grid/weights-gem2.5-ops.nc') # Created two files, one with each mask for the operational files time period
k = calculation(k)

# Fourth loop (23 Feb 2023 - 30 Apr 2024)
path = ('/results/forcing/atmospheric/continental2.5/nemo_forcing/')
folders = [x for x in os.listdir(path) if ((x[12:14]=='01' or (x[12:14]=='02' and x[15:17] < '29') or x[12:14]=='03' or x[12:14] == '04') and (x[7:11] < '2026'))]
folders.sort()
folders = folders[2:] # Cut the first two days in the folder
weight = ('/home/sallen/MEOPAR/grid/weights-continental2.5-hrdps_202108_23feb23onward.nc')
k = calculation(k)


## Calling the file creation

In [19]:
ds0.close()

file_creation(wind_speed, 'Mean_wind_speed')
file_creation(wind_direction_1, 'Mean_wind_direction_1')
file_creation(wind_direction_2, 'Mean_wind_direction_2')
file_creation(wind_velocity, 'Mean_wind_velocity')
